
`Shapespyer` is a Python toolkit that builds starting configurations for molecular dynamics (MD) simulations of soft matter, such as micelles, vesicles and bilayers in solvent, and equilibrates the systems with GROMACS. We're developing a desktop app `EasyShapes` that uses `shapes.project`, a Shapespyer package, as its backend for building and equilibrating such systems. The notebooks for our section today use `shapes.project` directly. It provides pythonic API that could be used in jupyter, python scripts and GUI, but the same functionality (and more) is available with Shapespyer's command-line tools (`smiles`, `shape`, `gmx-*`). While the command-line tools suit batch runs from shell scripts and cluster job queues, `shapes.project` suits interactive work.

#### Section program following the `EasyShapes` workflow and `shapes.project` layout

The notebooks of this workshop section follow the layout of `shapes.project`. In order to equilibrate a system, a project needs a sample model. A sample model, in turn, needs a structure and a buffer, and a structure needs components with their counter-ions. The notebooks build this up in the opposite direction, one step at a time, from left to right in the diagram:

```
┌─ Library ────────────────────────────────────────────────────────────────────────────────┐
│ Assets downloaded from Shapespyer repo or saved by the user                              │
└─────────────┬──▲───────────────────┬──▲────────────────────────┬──▲───────────────────┬──┘
          load│  │save           load│  │save                load│  │save           load│
              ▼  │                   ▼  │                        ▼  │                   ▼
┌─ Component ──────┐   ┌─ Structure ─────────┐   ┌─ SampleModel ──────┐   ╔═ Project ══════╗
│ .paths           │──►│ .components         │──►│ .structure         │──►║ .sample_model  ║
│ .counter_ion     │   │ .layers (Ball)      │   │ .buffer            │   ║ .analysis      ║
│ .mint            │   │ .lamellae (Vesicle) │   │                    │   ║                ║
│ .mext            │   │ .motif (Lattice)    │   │                    │   ║                ║
└──────────────────┘   └─────────────────────┘   └────────────────────┘   ╚════════════════╝
  notebook 1             notebook 2                notebook 3               notebook 4
└─────────────────── EasyShapes: Sample Model tab ────────────────────┘   Project + Analysis tabs
```

0. **Library** (all notebooks) is represented by the class `Library`, which manages the asset files, both the ready-made molecules and equilibration settings downloaded on first use and the assets you save. Every class that holds files has `load(name)` and `save()`, which use the `Library` object underneath.
1. **Component** (notebook 1) is represented by the classes `Lipid` and `Surfactant`. Components are loaded from the library or created from a structure file or a SMILES string.
2. **Structure** (notebook 2) is represented by the classes `Ring`, `Rod`, `Bilayer`, `Monolayer`, `Ball`, `Vesicle` and `Lattice`, each arranging copies of components in its shape.
3. **SampleModel** (notebook 3) is a structure in a buffer of solvent and salt, solvated with GROMACS. The buffer is represented by the classes `Buffer`, `Solvent` and `Salt`.
4. → **Project** (notebook 4) holds a sample model and its analysis, the staged GROMACS equilibration that relaxes the model.

# 4. Equilibration with GROMACS

This notebook works with a whole project, with a sample model and its equilibration.

```
                       ┌─ Library ──────┐
                       └─────────────┬──┘
                                 load│ configuration
                                     ▼
     ┌─────────────┐   ╔═ Project ══════╗
… ──►│ SampleModel │──►║ .sample_model  ║
     └─────────────┘   ║ .analysis      ║
                       ╚════════════════╝
```

We put an SDS micelle (a ball) in water, inside a project, and equilibrate it with GROMACS. The equilibration runs in stages:

- **stage 0**: energy minimisation, about 1–2 minutes on 2 CPU cores
- **stage 1**: short MD with position restraints, about 20 minutes on 2 CPU cores, if there is time.

Then we look at the outputs, save the project, and export the stages to open them in VMD.

We have to start with a few setup cells. Make sure Shapespyer is installed in your virtual environment, and GROMACS on your system (on WSL or Linux: `sudo apt install gromacs`). [Shapespyer installation docs](https://simnavi.gitlab.io/shapespyer.io/installation.html)

In [ ]:
# Setup: check that shapespyer.yaml is next to the notebook
from pathlib import Path

CONFIG_FILE = Path("shapespyer.yaml")  # where Shapespyer keeps its files
if not CONFIG_FILE.is_file():
    err_msg = f"No {CONFIG_FILE} in {Path.cwd()}; run the notebook in its own folder, next to shapespyer.yaml"
    raise FileNotFoundError(err_msg)

The next cell provides helper functions that only check the setup and show results.
These helper functions, as well as the helpers in other notebooks, **are not part of the Shapespyer workflow**. They are all in the `Helpers` class, and the notebook always calls them from it, as `Helpers.<name>()`, so you can tell them apart from the Shapespyer package.

In [ ]:
# Helpers (not part of the Shapespyer workflow)
import re

import matplotlib.pyplot as plt
import mdtraj
import nglview
import numpy as np


class Helpers:
    """Workshop helpers, not part of Shapespyer: call them as Helpers.<name>()."""

    @staticmethod
    def check_setup(config):
        """Stop with a message if GROMACS can't run; else print its version."""
        if config.gmx_problem is not None:
            err_msg = (
                f"This notebook needs GROMACS: {config.gmx_problem}. Install it, or set gmx_executable"
                f" in {config.file.name}. On Windows, the easy way is to run this notebook in WSL."
            )
            raise RuntimeError(err_msg)
        print("GROMACS:", config.gmx_version)

    @staticmethod
    def show_mdp_settings(configuration):
        """Print the main settings of each equil<k>.mdp, to compare the stages."""
        print(f"{'file':<12}{'integrator':<12}{'nsteps':>9}{'dt (ps)':>9}{'time (ps)':>11}  define")
        for mdp in sorted(configuration.paths):
            found = re.findall(r"^\s*([\w-]+)\s*=\s*([^;\n]*?)\s*(?:;.*)?$", mdp.read_text(), re.MULTILINE)
            settings = dict(found)
            nsteps, dt = settings.get("nsteps", ""), settings.get("dt", "")
            time_ps = f"{int(nsteps) * float(dt):g}" if nsteps and dt else ""
            integrator = settings.get("integrator", "")
            print(f"{mdp.name:<12}{integrator:<12}{nsteps:>9}{dt:>9}{time_ps:>11}  {settings.get('define', '')}")

    @staticmethod
    def show_minimisation_result(stage):
        """Print the lines of a minimisation stage's .log that tell whether it converged."""
        for line in Helpers._file_with_suffix(stage, ".log").read_text().splitlines():
            if "converged" in line or "Potential Energy" in line or "Maximum force" in line:
                print(line)

    @staticmethod
    def _file_with_suffix(asset, suffix):
        found = [path for path in asset.paths if path.suffix == suffix]
        if not found:
            raise FileNotFoundError(f"{asset.name} holds no {suffix} file yet")
        return found[0]

    @staticmethod
    def _molecules(topology, resname):
        """Atom indices of the residues named resname (a name or a list of names), or of all but the water."""
        # mdtraj renames TIP3 to HOH on loading, so select it with the "water" keyword
        if resname is None:
            return topology.select("not water")
        names = [resname] if isinstance(resname, str) else list(resname)
        atoms = topology.select(" or ".join(f"resname {name}" for name in names))
        if len(atoms) == 0:
            found = sorted({residue.name for residue in topology.residues})
            raise ValueError(f"No residues named {', '.join(names)}; the file has {', '.join(found)}")
        return atoms

    @staticmethod
    def show_without_water(asset, resname=None):
        """Draw the .gro an asset holds, such as a sample model or a stage, without the water.

        Give resname, a residue name or a list of them, to draw only those molecules.
        """
        frame = mdtraj.load(Helpers._file_with_suffix(asset, ".gro"))
        shown = frame.atom_slice(Helpers._molecules(frame.top, resname))
        print(f"{frame.n_atoms} atoms, box {frame.unitcell_lengths[0]} nm; {shown.n_atoms} shown")
        return nglview.show_mdtraj(shown)

    @staticmethod
    def _load_trajectory(stage):
        xtc = Helpers._file_with_suffix(stage, ".xtc")
        gro = Helpers._file_with_suffix(stage, ".gro")
        return mdtraj.load(xtc, top=gro)

    @staticmethod
    def show_trajectory_without_water(stage, resname=None):
        """Draw the trajectory of an MD stage, without the water; resname draws only those molecules."""
        trajectory = Helpers._load_trajectory(stage)
        shown = trajectory.atom_slice(Helpers._molecules(trajectory.top, resname))
        return nglview.show_mdtraj(shown)

    @staticmethod
    def plot_micelle_rg(stage, resname):
        """Plot the radius of gyration of the residues named resname, per frame of an MD stage."""
        trajectory = Helpers._load_trajectory(stage)
        micelle = trajectory.atom_slice(trajectory.top.select(f"resname {resname}"))
        masses = np.array([atom.element.mass for atom in micelle.top.atoms])
        rg = mdtraj.compute_rg(micelle, masses=masses)  # nm, one value per frame
        print(f"{micelle.n_frames} frames, Rg from {rg.min():.2f} to {rg.max():.2f} nm")
        plt.plot(micelle.time, rg)
        plt.xlabel("time (ps)")
        plt.ylabel("Rg of the micelle (nm)")
        plt.show()

    @staticmethod
    def show_stages(equilibration):
        """Print each stage of an equilibration with the names of its files."""
        for stage in equilibration.stages:
            print(stage.name, [path.name for path in stage.paths])

    @staticmethod
    def list_files(folder):
        """Print the files under a folder, relative to it."""
        for path in sorted(folder.rglob("*")):
            if path.is_file():
                print(path.relative_to(folder))

In the setup we pass the `shapespyer.yaml` config file, which says where Shapespyer keeps its files. Here it keeps the workshop's files apart from your own Shapespyer data. In the common scenario, users won't have to use the config file, as the `shapes.project` package manages files automatically. Use the config if you want manual control over where the files are kept.

In [ ]:
# Config file, log level and work folder
import time

from shapes.basics.utils import LogConfiguration
from shapes.project.files import AppFileManager

# the first call sets where the files go, for the whole session
config = AppFileManager(CONFIG_FILE).config
print("Workshop files:", config.app_path, "| projects:", config.projects_path)
Helpers.check_setup(config)

# the level applies to Shapespyer's own handlers; stop the messages there, so no other handler prints them again
LogConfiguration("WARNING").logger.propagate = False

# a new folder each run, so a project saved before is kept
DEMO_PATH = config.projects_path / f"DemoProj-{time.strftime('%Y%m%d-%H%M%S')}"
# the stage exports go to a work folder, which the clean up deletes
WORK = config.app_path / "work"

In [ ]:
from shapes.basics.defaults import Fill
from shapes.project import Ball, Ion, Library, Project, Surfactant
from shapes.project.analysis import AnalysisConfiguration
from shapes.project.species import Solvent

### The library

Here you choose where the library keeps its files: both the assets downloaded from the Shapespyer assets and the components you save yourself. We give `LocalAssets/` in the application directory, which is also the default. Only the first `Library()` of a session sets the path: later calls return the same library. `clean_temp()` deletes the working files that earlier runs left in Temp.

In [ ]:
library = Library(path=config.app_path / "LocalAssets")
library.clean_temp()

### Create the project and its sample model

A project holds:

- `title` and `description`: free text about the project.
- `path`: the folder the project is saved in. It goes into the projects folder that the config cell printed, `.data/Projects`. The path given to `Project` is only noted: the folder is made on the first save.
- `sample_model`: the model to work on, a new, empty one to start with.
- `analysis`: what is done with the model; its equilibration comes below.

`save()`, `save_as(path, overwrite=False)` and `Project.load(path)` save and load it, at the end of the notebook.

The sample model holds `name`, `structure` and `buffer`, as in notebook 3. The structure is an SDS ball, as in notebook 2, with the tails inside. The buffer holds only a solvent, TIP3 water.

SDS is charged, so it gets sodium (NAT) as its counter ion.

In [ ]:
proj = Project(
    title="SDS micelle",
    description="An SDS micelle in water, equilibrated with GROMACS",
    path=DEMO_PATH,
)
model = proj.sample_model
model.name = "SDS_ball"

ball = Ball(
    name="SDS_micelle",
    fill=Fill.RINGS0,
    fxz=False,
    rev=False,
)
ball.layers[0].dmin = 0.5
ball.layers[0].rmin = 1.2

sds = Surfactant.load("SDS")
sds.mint = 41
sds.mext = 0
sds.counter_ion = Ion.load("NAT")
ball.components.append(sds)

model.structure = ball
model.buffer.solvent = Solvent.load("TIP3")

### Assemble the sample model

The assembly builds the ball, puts in one Na⁺ per SDS with `gmx insert-molecules`, and fills the box with water with `gmx solvate`. The files are made in Temp, in the application directory.

In [ ]:
model.assemble()
for path in model.paths:
    print(path)

### Set up the equilibration

`proj.analysis` holds the model's equilibration. An equilibration holds:

- `configuration`: the `equil<k>.mdp` file of each stage, loaded from the library: `AnalysisConfiguration.load(name)`. The library has CHARMM36.
- `force_field`: the force field of the topologies. Unset, it loads CHARMM36 from the library.
- `first_stage` and `last_stage`: the first and the last stage of the next run. Unset, the last stage is the configuration's last.
- `threads`: the threads `mdrun` runs on. Unset, it uses the cores it may run on.
- `restrain_to_box_center`: anchors the position restraints of the structure's components at the box centre, instead of at their atoms' starting positions. Off by default.
- `stages`: the stages run so far, `stages[k]` being stage k.

`run()` runs the stages from `first_stage` to `last_stage` on the project's model, which the project links to its equilibration.

Stage *k* runs `equil<k>.mdp`, starting from the final `.gro` of stage *k − 1*. The library's CHARMM36 set has 7 stages:

In [ ]:
equilibration = proj.analysis.equilibration
equilibration.configuration = AnalysisConfiguration.load("CHARMM36")
Helpers.show_mdp_settings(equilibration.configuration)

### Stage 0: energy minimisation

`first_stage` and `last_stage` choose the stages to run. `.run()` waits until GROMACS ends.

In [ ]:
equilibration.first_stage = 0
equilibration.last_stage = 0
equilibration.run()

for path in equilibration.stages[0].paths:
    print(path)

A stage holds its `name`, `eq<k>`, and its `paths`, the files `gmx` wrote for it:

- `.tpr`: the run input, made by `gmx grompp`
- `.log`: the log of `gmx mdrun`
- `.edr`: energies
- `.trr`: trajectory (`.xtc` for MD stages)
- `.gro`: final coordinates; the next stage starts from them

Did the minimisation converge?

In [ ]:
Helpers.show_minimisation_result(equilibration.stages[0])

The minimised micelle. `resname="SDS"` draws only the SDS, without the water and the Na⁺ ions.

In [ ]:
Helpers.show_without_water(equilibration.stages[0], resname="SDS")

### Stage 1, if there is time: short MD

Stage 1 runs MD with position restraints. It takes about 20 minutes on 2 CPU cores, and the cell shows nothing until GROMACS ends. A run that ends is published to `equilibration.stages[1]`. Without the time, skip this cell and the trajectory and Rg cells after it. With *Run all*, the notebook waits here until stage 1 ends.

To stop it early, interrupt the cell: *Interrupt* in Jupyter or VS Code. That stops GROMACS too. Stage 0 keeps its files; skip the trajectory and Rg cells.

In [ ]:
equilibration.first_stage = 1
equilibration.last_stage = 1
equilibration.run()

for path in equilibration.stages[1].paths:
    print(path)

The stage 1 trajectory of the micelle, SDS only:

In [ ]:
Helpers.show_trajectory_without_water(equilibration.stages[1], resname="SDS")

How big is the micelle during stage 1? The radius of gyration (Rg), with the real masses of the atoms, gives one value per frame.

In [ ]:
Helpers.plot_micelle_rg(equilibration.stages[1], resname="SDS")

### Save the project

`save()` makes the project folder and copies into it the sample model, the analysis and every asset in them, with the stages. `ser_data.yaml` records the whole project, and `manifest.yaml` lists the files.

In [ ]:
proj.save()
Helpers.list_files(proj.path)

The library lists the saved projects, each with its last save time. `library.available_examples` lists the example projects that come with Shapespyer; loading one gives a new project of your own.

In [ ]:
library.available_projects

`Project.load` restores it from the folder, with the stages:

In [ ]:
loaded = Project.load(DEMO_PATH)
Helpers.show_stages(loaded.analysis.equilibration)

### Export the stages and open them in VMD

The saved project is for Shapespyer: load it to go on with it. To look at the stages in VMD, export them: `export_to()` copies the files of a stage into a folder you choose, here `stages/eq<k>/` in the work folder. Then the cell lists their files. Open a stage in VMD from its folder, for example:

```
vmd SDS_ball-eq0.gro
vmd SDS_ball-eq1.gro SDS_ball-eq1.xtc
```

In [ ]:
STAGE_EXPORTS = WORK / "stages"
for stage in equilibration.stages:
    stage.export_to(STAGE_EXPORTS / stage.name, overwrite=True)
Helpers.list_files(STAGE_EXPORTS)

### Clean up (optional)

This deletes the saved project, the work folder with the stage exports, and Temp, where the model and the stages were made. `force=True` is needed, because the assets of this session still hold their files there. Skip it to keep the project.

In [ ]:
import shutil

project_dir = proj.path
library.delete_project(project_dir, force=True)
shutil.rmtree(WORK, ignore_errors=True)
library.clean_temp(force=True)
print(project_dir.exists(), WORK.exists())